# Approach 1
This approach applies TCDB to acquire transporters and their TC identification (TCID). Subsequently, mechanisms for the relevant families are obtained manually, before they are connected to the reaction of the mechanism. This will in turn be crosschecked with Rhea, which is mapped to through UniProt IDs (UID).

Semantically, it will follow something along these lines: TCID + substrate + mechanism -> chemical reaction. Connect this and comparte with reaction on Rhea.

In [16]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd

The relevant files are as following:
1) "All proteins in TCDB (FASTA format)" - tc_fasta_url
2) "Tab-delimited table mapping TC systems to their substrates and ChEBI IDs" - tc_substrates_url

"Tab-delimited table mapping TC uniprot/refseq accessions to TC systems" is not necessary, as the UID is listed in the FASTA format file

In [17]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def parse_data(fasta_txt, substrates_txt):
    # FASTA (TCID and UID)
    fasta_data = [[record.description.split("|")[3].split()[0], record.description.split("|")[2]]
                  for record in SeqIO.parse(StringIO(fasta_txt), "fasta")]

    df_fasta = pd.DataFrame(fasta_data, columns=["TCID", "UID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
        for line in substrates_lines
        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])


    return df_fasta, df_substrates

In [18]:
tc_fasta_url = "https://www.tcdb.org/public/tcdb"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"

tc_fasta_txt = fetch_data(tc_fasta_url)
tc_substrates_txt = fetch_data(tc_substrates_url)

df_fasta, df_substrates = parse_data(tc_fasta_txt, tc_substrates_txt)

First I'll merge the DFs, before I reduce it, as only the top ten families in each of the subclasses below are of interest. These, alongside their general mechanism and acting entity can be obtained from Misc/All_comp/family_mechanisms_entity_all.tsv

The relevant subclasses were retrieved in Misc/TCDB_composition.ipynb, and are: [1.A, 1.B, 1.C, 2.A, 3.A]\
From these subclasses, the ten most populated familes were obtained for further analysis.

In [19]:
df = pd.merge(df_fasta, df_substrates, on="TCID", how="left")

# Importing the families and related mechanisms
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_all.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "⇌", "â†’": "→"}, regex=True)

# Filter out families not in top 10 of each of the selected subclasses
df["Family"] = df["TCID"].apply(lambda x: ".".join(x.split(".")[:3]))
df = df[df["Family"].isin(df_family_mechanisms["Family"])]
df = df.merge(df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left")
df = df.drop(columns=["Family"])

Now, many of the CHEBI IDs are secondary IDs, and needs to be converted in order to map to Rhea for cross-checking.

In [20]:
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
df["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))

In [21]:
df

,TCID,UID,CHEBI ID,CHEBI Name,Mechanism,Acting Entity
0,3.A.1.12.16,5IIP_A,CHEBI:15354,choline,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate
1,3.A.1.12.16,5IIP_A,CHEBI:3424,carnitinium,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate
2,3.A.1.12.16,5IIP_A,CHEBI:17750,glycine betaine,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate
3,3.A.1.12.16,5IIP_A,CHEBI:17203,L-proline,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate
4,1.A.9.5.14,5O8F_E,CHEBI:17996,chloride,ions (in) ⇌ ions (out),ions
...,...,...,...,...,...,...
12414,1.A.17.5.18,XP_723491.2,NaN,NaN,"Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...","Cl-, Cations"
12415,3.A.1.211.22,YP_009001498.1,NaN,NaN,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate
12416,2.A.7.1.17,YP_009173407.1,NaN,NaN,Nan,NaN
12417,1.A.1.7.8,YP_009174599.1,CHEBI:29103,potassium(1+),cation (out) ⇌ cation (in),cation


Future plan: Go through families_mechanisms_all.tsv and create another column for the acting entity in the reaction. The acting entity x will be marked as such: {x}\
This was first conudcted through AI, as this is a tedious manual task, before it was verified and edited by hand.

In [30]:
def create_reaction_row(row):

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    

    mechanisms = row["Mechanism"].split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = []
    for mechanism, entity in zip(mechanisms, acting_entities):
        reaction = mechanism.replace(entity, chebi_name)
        reactions.append(reaction)
    
    return ", ".join(reactions)

df["Reaction"] = df.apply(create_reaction_row, axis=1)

In [31]:
df

,TCID,UID,CHEBI ID,CHEBI Name,Mechanism,Acting Entity,Reaction
0,3.A.1.12.16,5IIP_A,CHEBI:15354,choline,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,choline (out) + ATP → choline (in) + ADP + Pi
1,3.A.1.12.16,5IIP_A,CHEBI:3424,carnitinium,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,carnitinium (out) + ATP → carnitinium (in) + A...
2,3.A.1.12.16,5IIP_A,CHEBI:17750,glycine betaine,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,glycine betaine (out) + ATP → glycine betaine ...
3,3.A.1.12.16,5IIP_A,CHEBI:17203,L-proline,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,L-proline (out) + ATP → L-proline (in) + ADP + Pi
4,1.A.9.5.14,5O8F_E,CHEBI:17996,chloride,ions (in) ⇌ ions (out),ions,chloride (in) ⇌ chloride (out)
...,...,...,...,...,...,...,...
12414,1.A.17.5.18,XP_723491.2,NaN,NaN,"Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ...","Cl-, Cations","Cl- (out) ⇌ Cl- (in), Cations (out) ⇌ Cations ..."
12415,3.A.1.211.22,YP_009001498.1,NaN,NaN,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,Substrate (out) + ATP → Substrate (in) + ADP + Pi
12416,2.A.7.1.17,YP_009173407.1,NaN,NaN,Nan,NaN,Nan
12417,1.A.1.7.8,YP_009174599.1,CHEBI:29103,potassium(1+),cation (out) ⇌ cation (in),cation,potassium(1+) (out) ⇌ potassium(1+) (in)


Alright, the case is. There are some minor issues with the reactions as of now, but that is mainly when there are two active entities in the reaction (e.g. Me1/Me2). However, this will be an issue to look into later, and now the next focus will be to obtain the Rhea reactions, including both the names and the ChEBI IDs. This will go through UniProt.